In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [2]:
df = pd.read_csv(r'C:\Users\vesna\Desktop\Neuefische\MyWork\WHO\WHO_health_indicators\WHS2025_DATADOWNLOAD.csv')

In [3]:
df.head()

,IndicatorName,IndicatorCode,Location,LocationCode,Year,Disaggregation,NumericValue,DisplayValue,Comments
0,Adolescent birth rate (per 1000 women aged 10-...,MDG_0000000003,Afghanistan,AFG,2021,AGEGROUP_YEARS10-14,18.000000,18.0,Afghanistan 2022-2023 Multiple Indicator Clust...
1,Adolescent birth rate (per 1000 women aged 10-...,MDG_0000000003,African Region,AFR,2024,AGEGROUP_YEARS10-14,2.991000,2.99,NaN
2,Adolescent birth rate (per 1000 women aged 10-...,MDG_0000000003,Albania,ALB,2022,AGEGROUP_YEARS10-14,0.210000,0.2,Registration Eurostat
3,Adolescent birth rate (per 1000 women aged 10-...,MDG_0000000003,Andorra,AND,2020,AGEGROUP_YEARS10-14,0.000000,0,Registration National Statistics & WPP2024
4,Adolescent birth rate (per 1000 women aged 10-...,MDG_0000000003,Antigua and Barbuda,ATG,2020,AGEGROUP_YEARS10-14,0.307977,0.3,Registration National Statistics & WPP2024


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9729 entries, 0 to 9728
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   IndicatorName   9729 non-null   str    
 1   IndicatorCode   9729 non-null   str    
 2   Location        9729 non-null   str    
 3   LocationCode    9729 non-null   str    
 4   Year            9729 non-null   str    
 5   Disaggregation  1517 non-null   str    
 6   NumericValue    9665 non-null   float64
 7   DisplayValue    9727 non-null   str    
 8   Comments        2722 non-null   str    
dtypes: float64(1), str(8)
memory usage: 684.2 KB


Years stored as strings! 

In [5]:
df["IndicatorName"].value_counts()

IndicatorName
Healthy life expectancy at birth (years)                                                                                                                                                   576
Life expectancy at birth (years)                                                                                                                                                           576
Tuberculosis incidence (per 100 000 population)                                                                                                                                            205
Road traffic mortality rate (per 100 000 population)                                                                                                                                       204
Annual mean concentrations of fine particulate matter (PM2.5) in cities (µg/m3)                                                                                                            202
Diphtheria-tetanus-pertussis (D

In [6]:
df["IndicatorName"].value_counts().sum()

np.int64(9729)

In [7]:
df["IndicatorName"].nunique()

55

In [8]:
print("Unique years:", sorted(df['Year'].unique()))
print("Unique locations:", df['Location'].nunique())

Unique years: ['2015', '2015-2016', '2016', '2016-2017', '2016-2018', '2017', '2017-2018', '2018', '2018-2019', '2019', '2019-2020', '2019-2021', '2020', '2021', '2021-2022', '2022', '2022-2023', '2023', '2023-2024', '2024']
Unique locations: 205


Problem 1: mixed year format (2015, 2015 - 2016, etc.) 
Problem 2: unique locations 205 (while WHO is listing 194 countries in the world)

In [9]:
#Building up search engine for question: "What are the indicators related to healthcare access and quality?"

keywords = ['UHC', 'doctor', 'nurs', 'pharmac', 'expenditure']  #incomplete words to catch variations

for key in keywords:
    matches = df[df['IndicatorName'].str.contains(key, case=False)]['IndicatorName'].unique()
    print(f"'{key}':")
    for m in matches:
        print(f"{m}")

'UHC':
UHC: Service coverage index
'doctor':
Density of medical doctors (per 10 000 population)
'nurs':
Density of nursing and midwifery personnel (per 10 000 population)
'pharmac':
Density of pharmacists (per 10 000 population)
'expenditure':
Domestic general government health expenditure (GGHE-D) as percentage of general government expenditure (GGE) (%)
Population with household expenditures on health > 10% of total household expenditure or income (%)
Population with household expenditures on health > 25% of total household expenditure or income (%)


In [10]:
#Define exact indicator names we need
indicators = ['UHC: Service coverage index','Density of medical doctors (per 10 000 population)','Density of nursing and midwifery personnel (per 10 000 population)','Density of pharmacists (per 10 000 population)', 'Domestic general government health expenditure (GGHE-D) as percentage of general government expenditure (GGE) (%)']

I stored the exact names of 5  indicators in a list so I can use it to precisely filter the dataset.

In [11]:
#Filter only my 5 indicators
df_q = df[df['IndicatorName'].isin(indicators)].copy()

In [ ]:
print(df_q.shape) #after applying filter 979 rows and 9 columns

(979, 9)


In [13]:
# Solving problem 1 - Check how much data is in ranges vs single years. The solution will depend on the results. 

year_types = df_q['Year'].astype(str).apply(lambda x: 'range' if '-' in x else 'single')

#Converted the Year column to text, even though in .info() we saw it as string, as a safety measure used the astype(str) method
#to ensure we can apply the lambda function correctly.
#Used apply with lambda function to check if there is a dash in the year value 
#If there is, label it as 'range', otherwise 'single'

#Let's see how many of each type we have:
print(year_types.value_counts())

Year
single    979
Name: count, dtype: int64


All 979 rows are single years — zero ranges found

Problem 1 solved: no year format cleaning needed for the 5 indicators

In [14]:
df_q['year_type'] = year_types
print(df_q.groupby(['IndicatorName', 'year_type']).size().unstack(fill_value=0))

year_type                                           single
IndicatorName                                             
Density of medical doctors (per 10 000 population)     196
Density of nursing and midwifery personnel (per...     199
Density of pharmacists (per 10 000 population)         183
Domestic general government health expenditure ...     200
UHC: Service coverage index                            201


Density of pharmacists (per 10 000 population) has fewer rows — to be investigated further (Problem 3 - Missing values) !!!

In [15]:
# Solving Problem 2 — 205 locations but world has ~194 countries
# First we look at ALL location names to identify non-countries

all_locations = df_q['Location'].unique()
print(f"Total unique locations: {len(all_locations)}")
print("\nAll locations alphabetically:")
for loc in sorted(all_locations):
    print(loc)

Total unique locations: 202

All locations alphabetically:
Afghanistan
African Region
Albania
Algeria
Andorra
Angola
Antigua and Barbuda
Argentina
Armenia
Australia
Austria
Azerbaijan
Bahamas
Bahrain
Bangladesh
Barbados
Belarus
Belgium
Belize
Benin
Bhutan
Bolivia (Plurinational State of)
Bosnia and Herzegovina
Botswana
Brazil
Brunei Darussalam
Bulgaria
Burkina Faso
Burundi
Cabo Verde
Cambodia
Cameroon
Canada
Central African Republic
Chad
Chile
China
Colombia
Comoros
Congo
Cook Islands
Costa Rica
Croatia
Cuba
Cyprus
Czechia
Côte d'Ivoire
Democratic People's Republic of Korea
Democratic Republic of the Congo
Denmark
Djibouti
Dominica
Dominican Republic
Eastern Mediterranean Region
Ecuador
Egypt
El Salvador
Equatorial Guinea
Eritrea
Estonia
Eswatini
Ethiopia
European Region
Fiji
Finland
France
Gabon
Gambia
Georgia
Germany
Ghana
Global
Greece
Grenada
Guatemala
Guinea
Guinea-Bissau
Guyana
Haiti
Honduras
Hungary
Iceland
India
Indonesia
Iran (Islamic Republic of)
Iraq
Ireland
Israel
Italy
Jam

Non-countries to remove (found by inspecting data manually):
- African Region
- Eastern Mediterranean Region  
- European Region
- Global
- Region of the Americas
- South-East Asia Region
- Western Pacific Region

In [17]:
non_countries = [
    'African Region', 'Eastern Mediterranean Region',
    'European Region', 'Global',
    'Region of the Americas', 'South-East Asia Region',
    'Western Pacific Region'
]

df_q = df_q[~df_q['Location'].isin(non_countries)]
print("After removing non-countries:", df_q.shape)

After removing non-countries: (944, 10)


In [18]:
print(df_q.columns.tolist())

['IndicatorName', 'IndicatorCode', 'Location', 'LocationCode', 'Year', 'Disaggregation', 'NumericValue', 'DisplayValue', 'Comments', 'year_type']


In [ ]:
df_q = df_q.drop(columns=['year_type'])
print("Shape:", df_q.shape) 
print("Columns:", df_q.columns.tolist())

Shape: (944, 9)
Columns: ['IndicatorName', 'IndicatorCode', 'Location', 'LocationCode', 'Year', 'Disaggregation', 'NumericValue', 'DisplayValue', 'Comments']


In [20]:
original_locations = df['Location'].unique()
current_locations = df_q['Location'].unique()

lost = set(original_locations) - set(current_locations)
print("Lost locations:", lost)

Lost locations: {'Global', 'Eastern Mediterranean Region', 'South-East Asia Region', 'Western Pacific Region', 'China, Hong Kong SAR', 'Region of the Americas', 'European Region', 'Puerto Rico', 'China, Macao SAR', 'African Region'}


In [27]:
print(f"Original locations in dataset: {df['Location'].nunique()}")
print(f"Locations after filter: {df_q['Location'].nunique()}")


Original locations in dataset: 205
Locations after filter: 195


Problem 2 - locations solved 
- Original dataset: 205 locations
- After indicator filter: 195 (WHO regions had no indicator data so disappeared automatically)
- After explicit non-country removal: 195 (confirmed, nothing extra removed)

3 territories lost (Hong Kong, Macao, Puerto Rico) - data limitation, assuming the data for these territories are under China and US

In [ ]:
# Reshape from long to wide format using pivot_table
df_q_wide = df_q.pivot_table(index=['Location', 'LocationCode'],columns='IndicatorName',values='NumericValue').reset_index()

# Clean up column names
df_q_wide.columns.name = None

# Rename columns to shorter names for easier use
df_q_wide = df_q_wide.rename(columns={
    'UHC: Service coverage index': 'UHC_index',
    'Density of medical doctors (per 10 000 population)': 'doctors',
    'Density of nursing and midwifery personnel (per 10 000 population)': 'nurses',
    'Density of pharmacists (per 10 000 population)': 'pharmacists',
    'Domestic general government health expenditure (GGHE-D) as percentage of general government expenditure (GGE) (%)': 'gov_health_exp'
})

print("Shape:", df_q_wide.shape)         
print("Columns:", df_q_wide.columns.tolist())
print("\nMissing values:")
print(df_q_wide.isnull().sum())
print("\nPreview:")
df_q_wide.head()

Shape: (195, 7)
Columns: ['Location', 'LocationCode', 'doctors', 'nurses', 'pharmacists', 'gov_health_exp', 'UHC_index']

Missing values:
Location           0
LocationCode       0
doctors            6
nurses             3
pharmacists       19
gov_health_exp     2
UHC_index          1
dtype: int64

Preview:


,Location,LocationCode,doctors,nurses,pharmacists,gov_health_exp,UHC_index
0,Afghanistan,AFG,3.17,5.49,0.32,1.126057,40.88461
1,Albania,ALB,18.79,58.21,10.80,9.193673,63.76895
2,Algeria,DZA,16.60,25.08,3.00,5.361368,74.11164
3,Andorra,AND,50.68,47.22,9.77,15.870918,78.86248
4,Angola,AGO,2.44,18.73,0.40,6.699635,36.72917


In [32]:
#Saving clean wide format dataset
df_q_wide.to_csv('WHO_UHC.csv', index=False)

THE END OF CLEANING 